# Temporal digit classifier

Same MNIST CSV flow as `digit_classifier.ipynb`, but inference runs in **delay** (temporal) space using `TemporalDigitClassifier` from [`temporal_digit_classifier.py`](temporal_digit_classifier.py). The reference notebook trains a NumPy MLP with **two** hidden layers (25→15); here we use **one** hidden layer and delay-domain normalization (see ASPLOS-style temporal arithmetic).

**Pipeline:** pixels → importance in (0,1] → delay `−log(importance)` → nLSE/nLDE ops → class delays → logits `−β·delay` → softmax probabilities.

```mermaid
flowchart LR
  pixels[Pixels]
  imp[Importance]
  delay[Delay]
  logits[Logits]
  probs[Softmax]
  pixels --> imp
  imp -->|minus_log| delay
  delay -->|nlse_nlde| delay
  delay -->|minus_beta| logits
  logits --> probs
```

In [ ]:
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from matplotlib import pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from models.temporal_digit_classifier import TemporalDigitClassifier

In [ ]:
data = pd.read_csv(ROOT / "datasets" / "mnist" / "train.csv")

In [ ]:
data.head()

In [ ]:
data = np.array(data)
m, n = data.shape
print(f"{m} training examples with {np.sqrt(n - 1)} by {np.sqrt(n - 1)} pixels.")

In [ ]:
np.random.shuffle(data)
X = data[:, 1:] / 255.0
y = data[:, 0].astype(np.int64)
split = round(X.shape[0] * 0.7)

x_train_np = X[:split]
y_train_np = y[:split]
x_dev_np = X[split:]
y_dev_np = y[split:]

x_train = torch.tensor(x_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
x_dev = torch.tensor(x_dev_np, dtype=torch.float32)
y_dev = torch.tensor(y_dev_np, dtype=torch.long)

## Device and constants

`max_terms` must exist in `C_VALUES` / `D_VALUES` (and E/F) inside the checkpoint.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

CONSTANTS_PATH = str(ROOT / "constants" / "orig_constants.pt")
MAX_TERMS = 10

## Explicit importance ↔ delay (one batch)

`forward()` applies the same steps internally. Here we unpack them for inspection.

In [ ]:
model = TemporalDigitClassifier(
    constants_path=CONSTANTS_PATH,
    max_terms=MAX_TERMS,
).to(device)

xb = x_train[:8].to(device)
imp = model.normalize_to_importance(xb)
delay = model.importance_to_delay(imp)
print("batch shape:", xb.shape)
print("importance min/max:", float(imp.min()), float(imp.max()))
print("delay min/max:", float(delay.min()), float(delay.max()))

## Model and training

Logits are `−output_beta × class_delay`; `CrossEntropyLoss` uses those logits. Interpreting back in importance space: `exp(−delay)` is a per-class score; softmax turns logits into a distribution (not the same as normalizing raw `exp(−delay)` across classes unless logits are proportional to `−delay`).

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

batch_size = 64
epochs = 10
n_train = x_train.shape[0]
total_batches = (n_train + batch_size - 1) // batch_size

start = time.time()
for epoch in range(epochs):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    batches = 0
    for i in range(0, n_train, batch_size):
        idx = perm[i : i + batch_size]
        xb = x_train[idx].to(device)
        yb = y_train[idx].to(device)
        _, logits = model(xb, return_logits=True)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        batches += 1
    print(
        f"epoch {epoch + 1}/{epochs}  loss={epoch_loss / batches:.4f}"
    )

print(f"Completed in {time.time() - start:.2f} s")

## Output path: delays → logits → probabilities

Compare `softmax(−β·delay)` with raw `exp(−delay)` as an unnormalized importance-like score.

In [ ]:
model.eval()
with torch.no_grad():
    one = x_train[:1].to(device)
    probs, logits = model(one, return_logits=True)
    # Recover class delays from logits (beta=1 default)
    out_delay = -logits / model.output_beta
    raw_imp = model.delay_to_importance(out_delay)
    print("logits (first sample):", logits[0].cpu().numpy())
    print("probs:", probs[0].cpu().numpy())
    print("exp(-delay) per class (not normalized to sum 1):", raw_imp[0].cpu().numpy())

In [ ]:
def accuracy(model, x, y, device, batch_size=256):
    model.eval()
    n = x.shape[0]
    correct = 0
    with torch.no_grad():
        for i in range(0, n, batch_size):
            xb = x[i : i + batch_size].to(device)
            yb = y[i : i + batch_size]
            pred = torch.argmax(model(xb), dim=1).cpu()
            correct += (pred == yb).sum().item()
    return correct / n


train_acc = accuracy(model, x_train, y_train, device) * 100
val_acc = accuracy(model, x_dev, y_dev, device) * 100
print(f"Training accuracy = {train_acc:.2f}%")
print(f"Validation accuracy = {val_acc:.2f}%")

In [ ]:
index = random.randint(0, x_train.shape[0] - 1)
test = x_train[index : index + 1].to(device)
label = int(y_train[index].item())

plt.figure(figsize=(2, 2))
plt.imshow(test.cpu().reshape(28, 28), cmap="gray", interpolation="nearest")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

model.eval()
with torch.no_grad():
    probabilities = model(test)[0].cpu().numpy()
percentages = probabilities * 100.0
guess = int(np.argmax(probabilities))

print(f"Guess: {guess}")
print("Predictions:")
for digit, pct in enumerate(percentages):
    print(f"{digit}: {pct:.2f}%")